# Capstone · An APAC multi-factor equity mandate

**Week 6 · Final portfolio case study**

**Mandate.** HK$2bn Asia-Pacific ex Japan equity, benchmarked to a market-cap index, 4% annual
tracking-error budget, 30% maximum annual one-way turnover, long-only, 5% maximum active weight per
position.

**In this template** the investable universe is the 25 size × book-to-market portfolios you have used all
course. If your firm gives you stock-level Hong Kong / APAC data, replace the universe in section 1 and
keep the rest of the structure.

**Data.** Monthly returns from the Kenneth R. French Data Library (Tuck School of Business, Dartmouth), *Asia Pacific ex Japan* region: Australia, **Hong Kong**, New Zealand and Singapore, July 1990 onwards. The setup cell downloads it once and caches it in `finex_data/`.

### Submission checklist
- [ ] Every section below runs top to bottom without errors
- [ ] Your trial log is complete (section 4) and the deflated Sharpe ratio is reported
- [ ] Expected tracking error ≤ 4% and turnover ≤ 30% are demonstrated, not asserted
- [ ] Your two-page memo and slides reference the numbers in this notebook
- [ ] Submit the notebook link, memo and slides on the course page

In [ ]:
# FinEx Learning · lab setup. Run this cell first; it only defines helpers.
# Data: Kenneth R. French Data Library (Tuck School of Business, Dartmouth),
# Asia Pacific ex Japan series (Australia, Hong Kong, New Zealand, Singapore),
# downloaded on first use and cached next to the notebook.
import hashlib
import io
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

FRENCH = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"
CACHE = Path("finex_data")
CACHE.mkdir(exist_ok=True)


def _lines(name):
    path = CACHE / f"{name}.csv"
    if not path.exists():
        with urllib.request.urlopen(f"{FRENCH}{name}_CSV.zip", timeout=60) as response:
            archive = zipfile.ZipFile(io.BytesIO(response.read()))
            path.write_bytes(archive.read(archive.namelist()[0]))
    return path.read_text(encoding="latin-1").splitlines()


def load_french(name, table=None, percent=True):
    """One monthly table from a French data library file, indexed by month.

    `table` is the title above the table (e.g. "Average Value Weighted Returns -- Monthly");
    None means the first monthly table. Returns are converted to decimals.
    """
    tables, title, current = {}, "", None
    for line in _lines(name):
        cells = [c.strip() for c in line.split(",")]
        if line.lstrip().startswith(","):
            current = []
            tables.setdefault(title or "first", (cells[1:], current))
            continue
        if current is not None and len(cells[0]) == 6 and cells[0].isdigit():
            current.append(cells)
            continue
        if line.strip():
            title, current = line.strip(), None
    key = table or next(k for k, (_, rows) in tables.items() if rows)
    header, rows = tables[key]
    frame = pd.DataFrame(
        [r[1:] for r in rows],
        columns=header,
        index=pd.PeriodIndex([r[0] for r in rows], freq="M"),
    ).astype(float).replace(-99.99, np.nan)
    return frame / 100 if percent else frame


def check(name, condition, hint):
    """Automatic check: passes quietly, or explains what to look at."""
    if not bool(condition):
        raise AssertionError(f"✗ {name}. Hint: {hint}")
    print(f"✓ {name}")


def completion_code(lab):
    code = hashlib.sha256(f"finex-quant-{lab}".encode()).hexdigest()[:6].upper()
    print(f"\nAll checks passed. Your completion code for the course page: FINEX-{lab.upper()}-{code}")


def annualise(returns):
    """Mean, volatility, Sharpe and t-stat of monthly excess returns, annualised."""
    r = returns.dropna()
    mean, vol = r.mean() * 12, r.std() * np.sqrt(12)
    return pd.Series({"mean": mean, "vol": vol, "sharpe": mean / vol,
                      "t_stat": r.mean() / r.std() * np.sqrt(len(r)), "months": len(r)})


print("Setup complete. Data source: Kenneth R. French Data Library · Asia Pacific ex Japan.")


## 1 · Universe and benchmark

In [ ]:
returns = load_french("Asia_Pacific_ex_Japan_25_Portfolios_ME_BE-ME", "Average Value Weighted Returns -- Monthly")
firms = load_french("Asia_Pacific_ex_Japan_25_Portfolios_ME_BE-ME", "Number of Firms in Portfolios", percent=False)
avg_size = load_french("Asia_Pacific_ex_Japan_25_Portfolios_ME_BE-ME", "Average Firm Size", percent=False)
cap = (firms * avg_size).shift(1)
benchmark_weights = cap.div(cap.sum(axis=1), axis=0)
TE_BUDGET, TURNOVER_LIMIT, MAX_ACTIVE = 0.04, 0.30, 0.05

## 2 · Signals (reuse Lab 2)

Build value, momentum and quality/low-vol signals point-in-time and combine them. State your weighting rule *before* looking at results.

In [ ]:
# TODO: signals and composite

## 3 · Risk model and portfolio construction (reuse Lab 3)

Shrunk covariance; maximise expected active return subject to the tracking-error, turnover and active-weight limits with `cvxpy`.

In [ ]:
# TODO: monthly optimisation loop

## 4 · Honest backtest (reuse Lab 4)

Log every variant you try in `trial_log`. Report gross and net performance, turnover, and the deflated Sharpe ratio.

In [ ]:
trial_log = []  # append a dict for every variant you run
# TODO: backtest

## 5 · Attribution and stress tests (reuse Lab 5)

Brinson by size segment, factor regression, and drawdowns in 2008, 2015 and 2020.

In [ ]:
# TODO: attribution and stress tests

## 6 · Recommendation

In five bullet points: the thesis, the evidence, the risk budget, the costs, and what would make you wrong.